In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd
import json
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer
import torch

c:\xai_6th_adv\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_path = pd.read_csv('soft_prompt/data/law/prompt_tuning_train_text.csv')
generated_train_path = pd.read_csv('inference_output/law/weak_train_50_tiny_llama-1.1b_523_prompt_3.csv', sep="\t")

non_labeled_corpus = []
with open(r'retrieve\datasets\raw\beir\law\corpus_filtered.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        non_labeled_corpus.append(json.loads(line))

generated_query = []
with open('inference_output/law/weak_queries_50_tiny_llama-1.1b_523_prompt_3.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        generated_query.append(json.loads(line))

In [4]:
real_query_text = []

for i in range(generated_train_path.shape[0]):
    for j in range(len(generated_query)):
        if str(generated_train_path['query-id'][i]) == str(generated_query[j]['_id']):
            real_query_text.append(generated_query[j]['text'])

generated_train_path['text_x'] = real_query_text

real_corpus_text = []

for i in range(generated_train_path.shape[0]):
    for j in range(len(non_labeled_corpus)):
        if str(generated_train_path['corpus-id'][i]) == str(non_labeled_corpus[j]['_id']):
            real_corpus_text.append(non_labeled_corpus[j]['text'])

generated_train_path['text_y'] = real_corpus_text

generated_train_path.head(2)

,query-id,corpus-id,score,text_x,text_y
0,5000001,2613,1,부령이 정하는 바에 의하여 보건소장으로 하여금 다음 각호의 사업을 하게 되어야 한다,가등기담보 등에 관한 법률 15조 제15조(담보가등기권리의 소멸) 담보가등기를 마친...
1,5000002,2614,1,부령이 정하는 바에 의하여 보건소장으로 하여금 다음 각호의 사업을 하게 된다 나에 ...,가사근로자의 고용개선 등에 관한 법률 11조 제3장 가사서비스의 제공 제11조(가사...


In [5]:
import os, json, time, hashlib, tempfile, shutil
from typing import Dict, Any, List, Optional
import numpy as np
import faiss, torch
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer

# ===== 예외 =====
class PrebuiltIndexNotFound(Exception): ...
class ArtifactMissing(Exception): ...

# ===== 유틸 =====
def _safe_makedirs(p: str): os.makedirs(p, exist_ok=True)

def _atomic_write_bytes(dst: str, data: bytes):
    d = os.path.dirname(dst); _safe_makedirs(d)
    fd, tmp = tempfile.mkstemp(dir=d)
    try:
        with os.fdopen(fd, "wb", buffering=0) as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, dst)
    except Exception:
        try: os.remove(tmp)
        finally: raise

def _save_json(path: str, obj: Any):
    _atomic_write_bytes(path, json.dumps(obj, ensure_ascii=False, indent=2).encode("utf-8"))

def _load_json(path: str) -> Any:
    if not os.path.exists(path): raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def _save_numpy_atomic(path: str, arr: np.ndarray):
    d = os.path.dirname(path); _safe_makedirs(d)
    fd, tmp = tempfile.mkstemp(dir=d)
    try:
        with os.fdopen(fd, "wb", buffering=0) as f:
            np.save(f, arr); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except Exception:
        try: os.remove(tmp)
        finally: raise

def _load_numpy(path: str) -> np.ndarray:
    if not os.path.exists(path): raise FileNotFoundError(path)
    return np.load(path)

def _save_faiss(path: str, index: faiss.Index):
    _safe_makedirs(os.path.dirname(path)); faiss.write_index(index, path)

def _load_faiss(path: str) -> faiss.Index:
    if not os.path.exists(path): raise FileNotFoundError(path)
    return faiss.read_index(path)

# 코퍼스 키: 문서+instruction만 반영 (모델/메서드는 변종에 둠)
def _hash_corpus(docs: List[str], instruction: bool) -> str:
    h = hashlib.sha256()
    h.update(b"inst1" if instruction else b"inst0")
    for d in docs:
        h.update(b"\x1e"); h.update(d.encode("utf-8"))
    return h.hexdigest()[:16]

def _paths_corpus(store_dir: str, corpus_key: str) -> Dict[str, str]:
    root = os.path.join(store_dir, corpus_key)
    return {
        "root": root,
        "corpus": os.path.join(root, "corpus.json"),
        "corpus_config": os.path.join(root, "corpus_config.json"),
        "bm25_tokens": os.path.join(root, "bm25_tokens.json"),
        "variants_dir": os.path.join(root, "variants"),
    }

def _paths_variant(store_dir: str, corpus_key: str, variant: str) -> Dict[str, str]:
    vd = os.path.join(store_dir, corpus_key, "variants", variant)
    return {
        "dir": vd,
        "config": os.path.join(vd, "config.json"),
        "dense_npy": os.path.join(vd, "dense_emb.npy"),
        "faiss_index": os.path.join(vd, "faiss.index"),
    }

# ===== 1) 생성/저장 =====
def build_and_save_index(
    *,
    docs: List[str],
    instruction: bool,
    variant: str,                 # ex) "bge-m3" | "sbert" | "other"
    embed_model_name: str,        # ex) "BAAI/bge-m3", "sentence-transformers/all-MiniLM-L6-v2", ...
    method: str,                  # "dense" | "faiss" | "bm25"
    store_dir: str = "embedded_docs",
    corpus_key: Optional[str] = None,   # 미지정 시 자동 생성
    overwrite_variant: bool = False,    # 같은 variant 덮어쓸지
    build_bm25_once: bool = False,      # BM25 토큰도 함께 만들고 싶으면 True
) -> str:
    # instruction 프리픽스 적용
    _docs = [f"Document: {d}" for d in docs] if instruction else list(docs)
    corpus_key = corpus_key or _hash_corpus(_docs, instruction)

    PC = _paths_corpus(store_dir, corpus_key)
    PV = _paths_variant(store_dir, corpus_key, variant)

    # 코퍼스 메타/문서 저장(없으면)
    _safe_makedirs(PC["root"])
    if not os.path.exists(PC["corpus"]):
        _save_json(PC["corpus"], _docs)
        _save_json(PC["corpus_config"], {
            "instruction": instruction,
            "created_at": int(time.time()),
            "version": 1
        })

    # BM25 토큰 생성(옵션, 중복 방지)
    if build_bm25_once and not os.path.exists(PC["bm25_tokens"]):
        tok = AutoTokenizer.from_pretrained(embed_model_name)
        tokenized = [tok.tokenize(d) for d in _docs]
        _save_json(PC["bm25_tokens"], tokenized)

    # 변종 생성
    if (os.path.exists(PV["config"]) or os.path.exists(PV["dense_npy"]) or os.path.exists(PV["faiss_index"])) and not overwrite_variant:
        return corpus_key  # 이미 있음

    _safe_makedirs(PV["dir"])

    if method == "bm25":
        # 변종에 별도 파일은 없음(코퍼스 공용 bm25_tokens 사용)
        _save_json(PV["config"], {
            "variant": variant, "embed_model_name": embed_model_name,
            "method": "bm25", "created_at": int(time.time())
        })
        return corpus_key

    if method not in {"dense", "faiss"}:
        raise ValueError("method must be one of {'bm25','dense','faiss'}")

    model = SentenceTransformer(embed_model_name)
    emb = model.encode(_docs, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    _save_numpy_atomic(PV["dense_npy"], emb)

    if method == "faiss":
        dim = emb.shape[1]
        index = faiss.IndexFlatIP(dim)
        index.add(emb)
        _save_faiss(PV["faiss_index"], index)

    _save_json(PV["config"], {
        "variant": variant,
        "embed_model_name": embed_model_name,
        "method": method,
        "dim": int(emb.shape[1]) if method in {"dense","faiss"} else None,
        "created_at": int(time.time())
    })
    return corpus_key

# ===== 2) 로드 =====
def load_index(
    *,
    store_dir: str,
    corpus_key: str,
    variant: str,
    use_gpu_for_faiss: bool = True
) -> Dict[str, Any]:
    PC = _paths_corpus(store_dir, corpus_key)
    PV = _paths_variant(store_dir, corpus_key, variant)
    if not os.path.exists(PC["corpus"]):
        raise PrebuiltIndexNotFound(f"Corpus not found: {PC['root']}")

    docs = _load_json(PC["corpus"])
    if not os.path.exists(PV["config"]):
        raise PrebuiltIndexNotFound(f"Variant not found: {PV['dir']}")

    vcfg = _load_json(PV["config"])
    method = vcfg["method"]

    if method == "bm25":
        if not os.path.exists(PC["bm25_tokens"]):
            raise ArtifactMissing("bm25_tokens.json missing for corpus")
        tokens = _load_json(PC["bm25_tokens"])
        tokenizer = AutoTokenizer.from_pretrained(vcfg["embed_model_name"])
        bm25 = BM25Okapi(tokens)
        return {"method": "bm25", "docs": docs, "bm25": bm25, "tokenizer": tokenizer}

    if method == "dense":
        if not os.path.exists(PV["dense_npy"]):
            raise ArtifactMissing("dense_emb.npy missing for variant")
        emb = _load_numpy(PV["dense_npy"]).astype("float32")
        model = SentenceTransformer(vcfg["embed_model_name"])
        return {"method": "dense", "docs": docs, "docs_embeddings": emb, "dense_model": model}

    if method == "faiss":
        if not (os.path.exists(PV["dense_npy"]) and os.path.exists(PV["faiss_index"])):
            raise ArtifactMissing("faiss.index or dense_emb.npy missing for variant")
        emb = _load_numpy(PV["dense_npy"]).astype("float32")
        index = _load_faiss(PV["faiss_index"])  # CPU
        if use_gpu_for_faiss and torch.cuda.is_available():
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
        model = SentenceTransformer(vcfg["embed_model_name"])
        return {"method": "faiss", "docs": docs, "docs_embeddings": emb, "dense_model": model, "faiss_index": index}

    raise ValueError(f"unknown method in variant config: {method}")

# ===== 3) 검색 =====
def search(query: str, k: int, handle: Dict[str, Any]):
    m = handle["method"]; docs = handle["docs"]
    if m == "bm25":
        tok = handle["tokenizer"]; bm25 = handle["bm25"]
        q_tokens = tok.tokenize(query)
        scores = bm25.get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        return [(docs[i], float(scores[i])) for i in idx]
    if m == "dense":
        model = handle["dense_model"]; emb = handle["docs_embeddings"]
        qv = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0].astype("float32")
        sims = emb @ qv
        idx = np.argsort(sims)[::-1][:k]
        return [(docs[i], float(sims[i])) for i in idx]
    if m == "faiss":
        model = handle["dense_model"]; index = handle["faiss_index"]
        qv = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        D, I = index.search(qv, k)
        return [(docs[int(i)], float(D[0, j])) for j, i in enumerate(I[0])]
    raise ValueError("unknown method in handle")


In [11]:
variants      = ["bge-m3", "ko-legal-sbert", "qwen3-embedding"]
model_list    = ["upskyy/bge-m3-korean", "woong0322/ko-legal-sbert-finetuned", "Day1Kim/Qwen3-Embedding-0.6B-Korean"]
method_list   = ["bm25", "dense", "faiss"]
corpus_key    = "law-v1"

# 확인: 아래 한 줄이 위를 즉시 덮어씌웁니다. 의도된 건지 점검하세요.
# docs = list(generated_train_path['text_y'])
docs = list(train_path['text_y'])  # 최종적으로 이걸 사용하게 됨

In [ ]:

base_variant = "bge-m3"
for method in method_list:
    # (모델 × 메서드) 조합으로 variant 분기
    variant_name = f"{base_variant}@{method}"

    build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
    build_and_save_index(
        docs=docs,
        instruction=False,
        variant=variant_name,
        embed_model_name="upskyy/bge-m3-korean",
        method=method,                  # "bm25" | "dense" | "faiss"
        store_dir="embedded_docs",
        corpus_key=corpus_key,
        overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
        build_bm25_once=build_bm25
    )


Token indices sequence length is longer than the specified maximum sequence length for this model (541 > 512). Running this sequence through the model will result in indexing errors


In [ ]:

base_variant = "ko-legal-sbert"
for method in method_list:
    # (모델 × 메서드) 조합으로 variant 분기
    variant_name = f"{base_variant}@{method}"

    build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
    build_and_save_index(
        docs=docs,
        instruction=True,
        variant=variant_name,
        embed_model_name= "woong0322/ko-legal-sbert-finetuned",
        method=method,                  # "bm25" | "dense" | "faiss"
        store_dir="embedded_docs",
        corpus_key=corpus_key,
        overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
        build_bm25_once=build_bm25
    )


KeyboardInterrupt: 

In [ ]:

base_variant = "qwen3-embedding"
for method in method_list:

    # (모델 × 메서드) 조합으로 variant 분기
    variant_name = f"{base_variant}@{method}"

    build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
    build_and_save_index(
        docs=docs,
        instruction=False,
        variant=variant_name,
        embed_model_name="Day1Kim/Qwen3-Embedding-0.6B-Korean",
        method=method,                  # "bm25" | "dense" | "faiss"
        store_dir="embedded_docs",
        corpus_key=corpus_key,
        overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
        build_bm25_once=build_bm25
    )


In [ ]:
# docs = list(generated_train_path['text_y'])
# docs = list(train_path['text_y'])

# model_list = ["woong0322/ko-legal-sbert-finetuned", "upskyy/bge-m3-korean", "Day1Kim/Qwen3-Embedding-0.6B-Korean"]
# model_name = model_list[1]

# method_list = ["bm25", "dense", "faiss"]
# method = method_list[1]
# instruction=False
# input = train_path['text_x'][0]

# for model_name in model_list:
#     for method in method_list:

#         if model_name=="upskyy/bge-m3-korean":
#             input = f"Query: {input}"
#             instruction=True
        
#         settings = retrieve_settings(
#             embed_model_name=model_name,
#             method=method,                # "bm25" | "dense" | "faiss"
#             instruction=instruction,
#             docs=docs,                  # 문자열 리스트
#             store_dir="embedded_docs",     # 로컬 저장 루트
#             cache_key=None,                # 지정 안 하면 자동 해시로 생성
#             force_recompute=False,         # True면 덮어쓰기 재계산
#             use_gpu_for_faiss=False         # GPU 가능 시 GPU로 올림
#         )

#         retrieved_output = results = search(input, k=15, method=method, settings=settings)
#         print(f"{method.upper()}: {retrieved_output}")

BM25: [('고용보험 및 산업재해보상보험의 보험료징수 등에 관한 법률 15조 제15조(보험료율의 특례) ① 대통령령으로 정하는 사업으로서 매년 9월 30일 현재 고용보험의 보험관계가 성립한 후 3년이 지난 사업의 경우에 그 해 9월 30일 이전 3년 동안의 그 실업급여 보험료에 대한 실업급여 금액의 비율이 대통령령으로 정하는 비율에 해당하는 경우에는 제14조제1항에도 불구하고 그 사업에 적용되는 실업급여 보험료율의 100분의 40의 범위에서 대통령령으로 정하는 기준에 따라 인상하거나 인하한 비율을 그 사업에 대한 다음 보험연도의 실업급여 보험료율로 할 수 있다. 제15조(보험료율의 특례) ② 대통령령으로 정하는 사업으로서 매년 6월 30일 현재 산재보험의 보험관계가 성립한 후 3년이 지난 사업의 경우에 그 해 6월 30일 이전 3년 동안의 산재보험료(제13조제5항제2호에 따른 산재보험료율을 곱한 금액은 제외한다)에 대한 산재보험급여 금액(「산업재해보상보험법」 제37조제1항제3호나목에 따른 업무상의 재해를 이유로 지급된 보험급여는 제외한다)의 비율이 대통령령으로 정하는 비율에 해당하는 경우에는 제14조제3항 및 제4항에도 불구하고 그 사업에 적용되는 제13조제5항제1호에 따른 산재보험료율의 100분의 50의 범위에서 사업 규모를 고려하여 대통령령으로 정하는 바에 따라 인상하거나 인하한 비율(이하 "개별실적요율"이라 한다)을 제13조제5항제2호에 따른 산재보험료율과 합하여 그 사업에 대한 다음 보험연도의 산재보험료율로 할 수 있다. <개정 2017.10.24, 2021.4.13> 제15조(보험료율의 특례) ③ 제2항에 따른 개별실적요율을 산정할 때 수급인ㆍ관계수급인(「산업안전보건법」 제2조제8호 및 제9호에 따른 수급인ㆍ관계수급인을 말한다. 이하 이 조에서 같다) 또는 파견사업주(「파견근로자보호 등에 관한 법률」 제2조제3호에 따른 파견사업주를 말한다. 이하 이 조에서 같다)의 근로자에게 발생한 업무상 재해가 다음 각 호의 어느 하나에 해당하는 재해인 경우에는

EOFError: No data left in file

In [8]:
# test 

docs = list(generated_train_path['text_y'])
docs = list(train_path['text_y'])

input = "하자심사는 어떻게 진행되나요사례를 들어보면 이해가 더 쉬울 것 같습니다사례를 들어보세요권리는 무엇인가요권리는 어떠한 것인가요권리는 부동산을 구입하거나 임대차 계약을 체결할 때 그 부동산을 이용할 수 있는 권리입니다 즉 부동산을 직접 이용하거나 임대주택의 임차인으로서 부동산을 이용할 수 있는 권리를 말합니다 그리고 부동산을 직접 이용하려는 사람뿐만 아니라 부동산을 빌려주려고 하는 사람도 부동산에 대한 권리하자심사는 어떻게 진행되나요사례를 들어보면 이해가 더 쉬울 것 같습니다사례를 들어보세요권리는 무엇인가요권리는 어떠한 것인가요권리는 부동산을 구입하거나 임대차 계약을 체결할 때 그 부동산을 이용할 수 있는 권리입니다 즉 부동산을 직접 이용하거나 임대주택의 임차인으로서 부동산을 이용할 수 있는 권리를 말합니다 그리고 부동산을 직접 이용하려는 사람뿐만 아니라 부동산을 빌려주려고 하는 사람도 부동산에 대한 권리"
input = f"Query: {input}"

settings = retrieve_settings(
    embed_model_name="upskyy/bge-m3-korean",
    method="bm25",          
    instruction=True,
    docs=docs,
    store_dir="embedded_docs",
    cache_key=None,
    force_recompute=False,     
    use_gpu_for_faiss=False    
)

retrieved_output = results = search(input, k=5, method=method, settings=settings)
print(f"{method.upper()}: {retrieved_output}")

RuntimeError: Dense artifacts not loaded.

In [ ]:
### 생성된 weak query가 실제로 연관된 문서를 잘 검색하는가?
'''
하자심사는 어떻게 진행되나요사례를 들어보면 이해가 더 쉬울 것 같습니다사례를 들어보세요권리는 무엇인가요권리는 어떠한 것인가요권리는 부동산을 구입하거나 임대차 계약을 체결할 때 그 부동산을 이용할 수 있는 권리입니다 즉 부동산을 직접 이용하거나 임대주택의 임차인으로서 부동산을 이용할 수 있는 권리를 말합니다 그리고 부동산을 직접 이용하려는 사람뿐만 아니라 부동산을 빌려주려고 하는 사람도 부동산에 대한 권리하자심사는 어떻게 진행되나요사례를 들어보면 이해가 더 쉬울 것 같습니다사례를 들어보세요권리는 무엇인가요권리는 어떠한 것인가요권리는 부동산을 구입하거나 임대차 계약을 체결할 때 그 부동산을 이용할 수 있는 권리입니다 즉 부동산을 직접 이용하거나 임대주택의 임차인으로서 부동산을 이용할 수 있는 권리를 말합니다 그리고 부동산을 직접 이용하려는 사람뿐만 아니라 부동산을 빌려주려고 하는 사람도 부동산에 대한 권리'''

"실내공기질 관리법 11조 제11조(오염물질 방출 건축자재의 사용제한 등)① 다중이용시설 또는 공동주택(「주택법」 제2조제22호에 따른 건강친화형 주택은 제외한다. 이하 이 조에서 같다)을 설치(기존 시설 또는 주택의 개수 및 보수를 포함한다. 이하 이 조에서 같다)하는 자는 다음 각 호의 어느 하나에 해당하는 건축자재를 사용하려는 경우 환경부장관이 관계 중앙행정기관의 장과 협의하여 환경부령으로 정하는 기준을 초과하지 아니하는 것으로 제2항에 따른 확인을 받고 제11조의6제1항에 따른 표지를 붙인 건축자재만을 사용하여야 한다. 1. 접착제 제11조(오염물질 방출 건축자재의 사용제한 등)① 다중이용시설 또는 공동주택(「주택법」 제2조제22호에 따른 건강친화형 주택은 제외한다. 이하 이 조에서 같다)을 설치(기존 시설 또는 주택의 개수 및 보수를 포함한다. 이하 이 조에서 같다)하는 자는 다음 각 호의 어느 하나에 해당하는 건축자재를 사용하려는 경우 환경부장관이 관계 중앙행정기관의 장과 협의하여 환경부령으로 정하는 기준을 초과하지 아니하는 것으로 제2항에 따른 확인을 받고 제11조의6제1항에 따른 표지를 붙인 건축자재만을 사용하여야 한다. 2. 페인트 제11조(오염물질 방출 건축자재의 사용제한 등)① 다중이용시설 또는 공동주택(「주택법」 제2조제22호에 따른 건강친화형 주택은 제외한다. 이하 이 조에서 같다)을 설치(기존 시설 또는 주택의 개수 및 보수를 포함한다. 이하 이 조에서 같다)하는 자는 다음 각 호의 어느 하나에 해당하는 건축자재를 사용하려는 경우 환경부장관이 관계 중앙행정기관의 장과 협의하여 환경부령으로 정하는 기준을 초과하지 아니하는 것으로 제2항에 따른 확인을 받고 제11조의6제1항에 따른 표지를 붙인 건축자재만을 사용하여야 한다. 3. 실란트(sealant) 제11조(오염물질 방출 건축자재의 사용제한 등)① 다중이용시설 또는 공동주택(「주택법」 제2조제22호에 따른 건강친화형 주택은 제외한다. 이하 이 조에서 같다)을 설치(기존 시설 또는 주택의 개수 및 보수를 포함한다. 이하 이 조에서 같다)하는 자는 다음 각 호의 어느 하나에 해당하는 건축자재를 사용하려는 경우 환경부장관이 관계 중앙행정기관의 장과 협의하여 환경부령으로 정하는 기준을 초과하지 아니하는 것으로 제2항에 따른 확인을 받고 제11조의6제1항에 따른 표지를 붙인 건축자재만을 사용하여야 한다. 4. 퍼티(putty) 제11조(오염물질 방출 건축자재의 사용제한 등)① 다중이용시설 또는 공동주택(「주택법」 제2조제22호에 따른 건강친화형 주택은 제외한다. 이하 이 조에서 같다)을 설치(기존 시설 또는 주택의 개수 및 보수를 포함한다. 이하 이 조에서 같다)하는 자는 다음 각 호의 어느 하나에 해당하는 건축자재를 사용하려는 경우 환경부장관이 관계 중앙행정기관의 장과 협의하여 환경부령으로 정하는 기준을 초과하지 아니하는 것으로 제2항에 따른 확인을 받고 제11조의6제1항에 따른 표지를 붙인 건축자재만을 사용하여야 한다. 5. 벽지 제11조(오염물질 방출 건축자재의 사용제한 등)① 다중이용시설 또는 공동주택(「주택법」 제2조제22호에 따른 건강친화형 주택은 제외한다. 이하 이 조에서 같다)을 설치(기존 시설 또는 주택의 개수 및 보수를 포함한다. 이하 이 조에서 같다)하는 자는 다음 각 호의 어느 하나에 해당하는 건축자재를 사용하려는 경우 환경부장관이 관계 중앙행정기관의 장과 협의하여 환경부령으로 정하는 기준을 초과하지 아니하는 것으로 제2항에 따른 확인을 받고 제11조의6제1항에 따른 표지를 붙인 건축자재만을 사용하여야 한다. 6. 바닥재 제11조(오염물질 방출 건축자재의 사용제한 등)① 다중이용시설 또는 공동주택(「주택법」 제2조제22호에 따른 건강친화형 주택은 제외한다. 이하 이 조에서 같다)을 설치(기존 시설 또는 주택의 개수 및 보수를 포함한다. 이하 이 조에서 같다)하는 자는 다음 각 호의 어느 하나에 해당하는 건축자재를 사용하려는 경우 환경부장관이 관계 중앙행정기관의 장과 협의하여 환경부령으로 정하는 기준을 초과하지 아니하는 것으로 제2항에 따른 확인을 받고 제11조의6제1항에 따른 표지를 붙인 건축자재만을 사용하여야 한다. 7. 그 밖에 건축물 내부에 사용되는 건축자재로서 표면가공 목질판상(木質板狀) 제품 등 환경부령으로 정하는 것 제11조(오염물질 방출 건축자재의 사용제한 등)② 제1항 각 호의 건축자재를 제조하거나 수입하는 자는 그 건축자재가 제1항에 따른 기준을 초과하여 오염물질을 방출하는지 여부를 제11조의2에 따른 시험기관에서 확인받은 후 다중이용시설 또는 공동주택을 설치하는 자에게 공급하여야 한다. 다만, 다른 법령에 따라 이 법에 준하는 확인을 받은 경우 등 대통령령으로 정하는 경우에는 본문에 따른 확인을 받지 아니하고 건축자재를 공급할 수 있다.  제11조(오염물질 방출 건축자재의 사용제한 등)③ 환경부장관은 제13조제4항에 따라 오염물질을 채취ㆍ검사한 결과 제1항에 따른 기준을 초과하는 건축자재의 경우 제2항에 따른 시험기관에 확인의 취소를 명할 수 있으며, 시험기관의 장은 특별한 사유가 없으면 확인을 취소하여야 한다.  제11조(오염물질 방출 건축자재의 사용제한 등)④ 환경부장관은 제3항에 따라 확인이 취소된 건축자재 및 제11조의6제1항을 위반하여 표지를 붙인 건축자재의 제조자 또는 수입자에게 회수 등의 조치를 명하거나 해당 건축자재와 관련된 내용을 대통령령으로 정하는 바에 따라 공표할 수 있다.  제11조(오염물질 방출 건축자재의 사용제한 등)⑤ 제3항에 따른 확인의 취소, 제4항에 따른 회수 등의 조치명령 및 공표에 필요한 사항은 대통령령으로 정한다.  제11조(오염물질 방출 건축자재의 사용제한 등)⑥ 제2항에 따른 확인의 절차ㆍ방법 및 유효기간 등에 관하여 필요한 사항은 대통령령으로 정한다.  제11조(오염물질 방출 건축자재의 사용제한 등)⑦ 제2항에 따라 시험기관이 확인을 한 경우에는 환경부령으로 정하는 바에 따라 그 기록을 보관하여야 한다.  제11조의2(건축자재 오염물질 방출 확인 시험기관의 지정 등)① 환경부장관은 제11조제2항에 따라 건축자재의 오염물질 방출 여부를 확인할 수 있는 시험기관(이하 \"시험기관\"이라 한다)을 지정할 수 있다. 제11조의2(건축자재 오염물질 방출 확인 시험기관의 지정 등)② 시험기관으로 지정을 받으려는 자는 환경부령으로 정하는 시설ㆍ장비 및 기술인력 등의 요건을 갖추어야 한다. 제11조의2(건축자재 오염물질 방출 확인 시험기관의 지정 등)③ 제1항에 따라 지정을 받은 자가 환경부령으로 정하는 중요한 사항을 변경하려는 경우 변경신청을 하여야 한다. 제11조의2(건축자재 오염물질 방출 확인 시험기관의 지정 등)④ 시험기관의 지정요건, 지정절차 등에 필요한 사항은 환경부령으로 정한다. 제11조의3(시험기관 지정의 결격사유) 다음 각 호의 어느 하나에 해당하는 자는 제11조의2제2항에 따른 시험기관으로 지정받을 수 없다.1. 피성년후견인 또는 피한정후견인 제11조의3(시험기관 지정의 결격사유) 다음 각 호의 어느 하나에 해당하는 자는 제11조의2제2항에 따른 시험기관으로 지정받을 수 없다.2. 파산선고를 받고 복권되지 아니한 자 제11조의3(시험기관 지정의 결격사유) 다음 각 호의 어느 하나에 해당하는 자는 제11조의2제2항에 따른 시험기관으로 지정받을 수 없다.3. 이 법을 위반하여 징역의 실형을 선고받고 그 집행이 끝나거나(집행이 끝난 것으로 보는 경우를 포함한다) 집행을 받지 아니하기로 확정된 날부터 2년이 지나지 아니한 자 제11조의3(시험기관 지정의 결격사유) 다음 각 호의 어느 하나에 해당하는 자는 제11조의2제2항에 따른 시험기관으로 지정받을 수 없다.4. 제11조의4에 따라 지정이 취소(이 조 제1호 및 제2호에 해당하여 지정이 취소된 경우는 제외한다)된 후 2년이 지나지 아니한 자 제11조의3(시험기관 지정의 결격사유) 다음 각 호의 어느 하나에 해당하는 자는 제11조의2제2항에 따른 시험기관으로 지정받을 수 없다.5. 임원 또는 기관의 대표자 중에 제1호부터 제4호까지의 규정 중 어느 하나에 해당하는 자가 있는 법인 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.1. 거짓이나 그 밖의 부정한 방법으로 제11조의2에 따른 시험기관으로 지정을 받은 경우 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.2. 거짓이나 그 밖의 부정한 방법으로 시험기관 업무를 수행한 경우 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.3. 제11조의3제1호부터 제5호까지에 해당하게 된 경우. 다만, 제11조의3제5호에 해당하는 법인의 경우 해당 임원이나 대표자를 6개월 이내에 바꾸어 임명하는 경우는 제외한다. 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.4. 업무정지 기간에 시험기관 업무를 수행한 경우 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.5. 제11조제6항에 따른 확인의 절차ㆍ방법이나 제11조의5제1항에 따른 준수사항을 지키지 아니한 경우 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.6. 제11조의2제1항에 따라 지정을 받은 후 1년 이내에 업무를 개시하지 아니하거나 정당한 사유 없이 1년 이상 휴업한 경우 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.7. 제11조의2제2항에 따른 시설, 장비 및 기술인력 기준에 미달된 경우 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.8. 제11조의2제3항에 따른 변경신청을 하지 아니하거나 거짓 또는 부정한 방법으로 변경신청을 한 경우 제11조의4(시험기관의 지정 취소 등)① 환경부장관은 시험기관이 다음 각 호의 어느 하나에 해당하는 경우 지정을 취소하거나 1년 이내의 기간을 정하여 시험기관 업무의 정지를 명할 수 있다. 다만, 제1호부터 제3호까지에 해당하는 경우에는 지정을 취소하여야 한다.9. 제11조의5제2항에 따른 평가 기준에 미달된 경우 제11조의4(시험기관의 지정 취소 등)② 제1항에 따른 행정처분의 세부기준은 환경부령으로 정한다. 제11조의5(시험기관의 준수사항 등)① 시험기관은 확인시험 방법, 검사결과의 기록ㆍ보존 등 환경부령으로 정하는 준수사항을 지켜야 한다. 제11조의5(시험기관의 준수사항 등)② 환경부장관은 시험기관에 대하여 확인의 시험에 관한 능력을 평가할 수 있다. 제11조의5(시험기관의 준수사항 등)③ 제2항에 따른 평가에 필요한 사항은 환경부령으로 정한다. 제11조의6(건축자재의 표지)① 건축자재를 제조하거나 수입하는 자는 제11조제2항에 따라 해당 건축자재가 방출기준을 초과하지 아니한다고 확인받은 경우 및 다른 법령에 따라 이 법에 준하는 확인을 받은 경우에는 환경부령으로 정하는 바에 따라 이를 증명하는 표지를 붙여야 한다.  제11조의6(건축자재의 표지)② 제11조제2항 본문에 따라 확인을 받지 아니하거나 같은 조 제3항에 따라 확인이 취소된 건축자재 등 제1항에 해당되지 아니하는 건축자재는 동 표지를 사용하여서는 아니 된다. 제11조의7(실내라돈조사의 실시)① 환경부장관은 라돈(radon)의 실내 유입으로 인한 건강피해를 줄이기 위하여 실내공기 중 라돈의 농도 등에 관한 조사(이하 \"실내라돈조사\"라 한다)를 실시할 수 있다. 제11조의7(실내라돈조사의 실시)② 환경부장관은 실내라돈조사를 실시하려는 경우에는 그 조사의 목적ㆍ대상ㆍ방법 및 기간 등 조사에 필요한 사항을 환경부령으로 정하는 바에 따라 공고하여야 한다. 제11조의7(실내라돈조사의 실시)③ 환경부장관은 특정 지역에 대하여 실내라돈조사가 필요한 경우에는 해당 지역을 관할하는 시ㆍ도지사에게 그 조사를 실시하게 할 수 있다. 제11조의7(실내라돈조사의 실시)④ 시ㆍ도지사는 제3항에 따라 실내라돈조사를 실시한 경우에는 그 결과를 환경부장관에게 보고하여야 한다. 제11조의7(실내라돈조사의 실시)⑤ 환경부장관은 시ㆍ도지사에게 제3항에 따른 실내라돈조사에 필요한 기술적ㆍ행정적ㆍ재정적 지원을 할 수 있다. 제11조의8(라돈지도의 작성)① 환경부장관은 실내라돈조사의 실시 결과를 기초로 실내공기 중 라돈의 농도 등을 나타내는 지도(이하 \"라돈지도\"라 한다)를 작성할 수 있다. 제11조의8(라돈지도의 작성)② 라돈지도의 작성기준, 작성방법 및 제공 등에 필요한 사항은 환경부령으로 정한다. 제11조의9(라돈관리계획의 수립ㆍ시행 등)① 환경부장관은 실내라돈조사의 실시 및 라돈지도의 작성 결과를 기초로 라돈으로 인한 건강피해가 우려되는 시ㆍ도가 있는 경우 「환경보건법」 제9조에 따른 환경보건위원회의 심의를 거쳐 해당 시ㆍ도지사에게 5년마다 라돈관리계획(이하 \"관리계획\"이라 한다)을 수립하여 시행하도록 요청할 수 있다. 이 경우 시ㆍ도지사는 특별한 사유가 없으면 지역주민들의 의견을 들어 관리계획을 수립하여야 한다. 제11조의9(라돈관리계획의 수립ㆍ시행 등)② 관리계획에는 다음 각 호의 사항이 포함되어야 한다.1. 다중이용시설 및 공동주택 등의 현황 제11조의9(라돈관리계획의 수립ㆍ시행 등)② 관리계획에는 다음 각 호의 사항이 포함되어야 한다.2. 라돈으로 인한 실내공기오염 및 건강피해의 방지 대책 제11조의9(라돈관리계획의 수립ㆍ시행 등)② 관리계획에는 다음 각 호의 사항이 포함되어야 한다.3. 라돈의 실내 유입 차단을 위한 시설 개량에 관한 사항 제11조의9(라돈관리계획의 수립ㆍ시행 등)② 관리계획에는 다음 각 호의 사항이 포함되어야 한다.4. 그 밖에 라돈관리를 위하여 시ㆍ도지사가 필요하다고 인정하는 사항 제11조의9(라돈관리계획의 수립ㆍ시행 등)③ 시ㆍ도지사는 관리계획을 수립한 경우 그 내용 및 연차별 추진실적을 대통령령으로 정하는 바에 따라 환경부장관에게 보고하여야 한다. 제11조의9(라돈관리계획의 수립ㆍ시행 등)④ 환경부장관은 시ㆍ도지사에게 관리계획의 시행에 필요한 기술적ㆍ행정적ㆍ재정적 지원을 할 수 있다. 제11조의10(라돈저감공법의 사용 등 권고)① 시ㆍ도지사는 해당 시ㆍ도 내에서 라돈으로 인하여 건강상 위해가 우려되는 지역이 있는 경우에는 그 지역에서 다중이용시설 또는 공동주택 등을 설치(기존 시설 또는 주택 등의 개수 및 보수를 포함한다)하는 자에게 라돈의 실내 유입을 줄이기 위한 공법을 사용하는 등의 필요한 조치를 하도록 권고할 수 있다. 제11조의10(라돈저감공법의 사용 등 권고)② 시ㆍ도지사는 해당 시ㆍ도 내 라돈 농도가 높은 다중이용시설 또는 공동주택 등의 소유자등에게 실내 라돈 농도를 환경부령으로 정하는 기준에 맞게 관리하도록 권고할 수 있다."




In [ ]:
# DOCU
docs = list(generated_train_path['text_y'])

docs = list(train_path['text_y'])[:100]

In [ ]:
# DOCU
docs = list(generated_train_path['text_y'])

docs = list(train_path['text_y'])[:100]

In [18]:
train_path['text_y'][0]

'공동주택관리법 시행령 42조 제42조(하자보수보증금의 범위) ① 법 제38조제1항에 따라 예치하여야 하는 하자보수보증금은 다음 각 호의 구분에 따른 금액으로 한다. 1. 「주택법」 제15조에 따른 대지조성사업계획과 주택사업계획승인을 함께 받아 대지조성과 함께 공동주택을 건설하는 경우: 가목의 비용에서 나목의 가격을 뺀 금액의 100분의 3 제42조(하자보수보증금의 범위) ① 법 제38조제1항에 따라 예치하여야 하는 하자보수보증금은 다음 각 호의 구분에 따른 금액으로 한다. 2. 「주택법」 제15조에 따른 주택사업계획승인만을 받아 대지조성 없이 공동주택을 건설하는 경우: 사업계획승인서에 기재된 해당 공동주택의 총사업비에서 대지가격을 뺀 금액의 100분의 3 제42조(하자보수보증금의 범위) ① 법 제38조제1항에 따라 예치하여야 하는 하자보수보증금은 다음 각 호의 구분에 따른 금액으로 한다. 3. 법 제35조제1항제2호에 따라 공동주택을 증축ㆍ개축ㆍ대수선하는 경우 또는 「주택법」 제66조에 따른 리모델링을 하는 경우: 허가신청서 또는 신고서에 기재된 해당 공동주택 총사업비의 100분의 3 제42조(하자보수보증금의 범위) ① 법 제38조제1항에 따라 예치하여야 하는 하자보수보증금은 다음 각 호의 구분에 따른 금액으로 한다. 4. 「건축법」 제11조에 따른 건축허가를 받아 분양을 목적으로 공동주택을 건설하는 경우: 사용승인을 신청할 당시의 「공공주택 특별법 시행령」 제56조제7항에 따른 공공건설임대주택 분양전환가격의 산정기준에 따른 표준건축비를 적용하여 산출한 건축비의 100분의 3 제42조(하자보수보증금의 범위) ② 제1항에도 불구하고 건설임대주택이 분양전환되는 경우의 하자보수보증금은 제1항제1호 또는 제2호에 따른 금액에 건설임대주택 세대 중 분양전환을 하는 세대의 비율을 곱한 금액으로 한다.'

In [63]:
train_path['text_x'][0]

'배달앱종사자인데 배달 중 사고를 당했습니다. 산재처리를 받을 수 있을까요?'

In [ ]:
# !pip install rank-bm25
# !pip install faiss-cpu  (faiss-gpu)

In [ ]:
import math
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation

# 1) 데이터: (text1, text2, label)  label ∈ {0,1}
train_pairs = [
    ("근로시간은 1주 40시간을 초과할 수 없다.", "1주 40시간 이상 근무하면 안 된다.", 1),
    ("연장근로는 1주 12시간까지 가능하다.", "주당 연장근로 제한은 12시간이다.", 1),
    ("연장근로는 1주 12시간까지 가능하다.", "휴게시간은 근로시간 도중에 주어야 한다.", 0),
    # ... 여기에 실제 학습 데이터(양/음성) 추가
]

dev_pairs = [
    ("근로시간 단축", "주당 근무시간 감소", 1),
    ("연장근로", "연차휴가", 0),
    # ... 검증용
]

# 2) 모델 로드
model_name = "woong0322/ko-legal-sbert-finetuned"
model = SentenceTransformer(model_name)

# 3) DataLoader
train_examples = [InputExample(texts=[a, b], label=int(y)) for a, b, y in train_pairs]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

# 4) Loss: Contrastive (라벨 0/1)
#   distance_metric 기본은 Cosine로 동작 (내부적으로 1 - cos_sim 사용)
#   margin은 보통 0.5~0.7 사이에서 시작해 튜닝
train_loss = losses.ContrastiveLoss(model, margin=0.5)

# 5) 평가 지표: BinaryClassificationEvaluator (0/1 라벨)
#   sentence-transformers에 포함되어 있음
dev_sents1 = [a for a, _, _ in dev_pairs]
dev_sents2 = [b for _, b, _ in dev_pairs]
dev_labels = [int(y) for _, _, y in dev_pairs]
evaluator = evaluation.BinaryClassificationEvaluator(
    dev_sents1, dev_sents2, dev_labels, name="dev-bin"
)

# 6) 학습
num_epochs = 3
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,            # 없애도 됨
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    evaluation_steps=200,           # 데이터 크기에 맞게 조정
    output_path="./ko-legal-sbert-contrastive-01"
)

# 7) 사용 예시
from sentence_transformers import util
m = SentenceTransformer("./ko-legal-sbert-contrastive-01")
q = "근로시간 단축"
c = "주당 근무시간 감소"
sim = util.cos_sim(m.encode(q), m.encode(c)).item()
print("cosine:", sim)
# 임계값은 dev셋에서 ROC로 최적화 (예: 0.5~0.7 구간에서 탐색)
